In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

CREATE VOLUME IF NOT EXISTS workspace.bronze.landing;

In [0]:
%sql 

CREATE TABLE IF NOT EXISTS workspace.silver.households;
CREATE TABLE IF NOT EXISTS workspace.silver.readings;
CREATE TABLE IF NOT EXISTS workspace.silver.members;

In [0]:
from pyspark.sql import functions as F

df = (spark.read
      .option("header", "true")
      .option("quote", '"')
      .option("escape", '"')        
      .option("multiLine", "true")  
      .csv("/Volumes/workspace/bronze/landing/households.csv"))

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.bronze.households")

#cleaning household table

df_households = spark.sql("SELECT * from workspace.bronze.households")
df_households = df_households.withColumn("household_id", F.col("household_id").cast("int"))
df_households = df_households.dropDuplicates(["household_id"])
df_households = df_households.withColumn("size", F.col("size").cast("int"))
df_households = df_households.withColumnRenamed("size", "household_size")
df_households = df_households.withColumn("weekly_limit_kwh", F.col("weekly_limit_kwh").cast("double"))
df_households = df_households.withColumn("monthly_limit_kwh", F.col("monthly_limit_kwh").cast("double"))
df_households = df_households.drop("members")

df_households.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.households")

#creating the members table

df = spark.sql("SELECT * from workspace.bronze.households")
df.dropDuplicates(["household_id"])
df = df.withColumn("household_id", F.col("household_id").cast("int"))
df = df.withColumn("weekly_limit_kwh", F.col("weekly_limit_kwh").cast("double"))
df = df.withColumn("monthly_limit_kwh", F.col("monthly_limit_kwh").cast("double"))
df_members = df["household_id","members"]

member_schema = "array<struct<id:string,email:string,name:string,role:string>>"

df_members = (df_members
    .withColumn("m", F.explode(F.from_json("members", member_schema)))
    .select(
        "household_id",
        F.col("m.id").alias("member_id"),
        F.col("m.email").alias("email"),
        F.col("m.name").alias("name"),
        F.col("m.role").alias("role"),
    ))

df_members.write.mode("overwrite").option("mergeSchema","true").saveAsTable("workspace.silver.members")

member_counts = (df_members
    .groupBy("household_id")
    .agg(F.count("*").cast("int").alias("household_size")))

df.write.mode("overwrite").option("mergeSchema","true").saveAsTable("workspace.silver.households")